# SVM deconvolution

The goal is to test the use of SVMs to deconvolve the cell type composition
(using as input the matrix containing one line per cell-type specific dmr regions and one column per target cell type:
at (i,j) the value is the mean probit for cell type j averaged on all the reads that overlap a dmr region specific to cell type j)

In [ ]:
import os
import warnings
from contextlib import contextmanager


import numpy as np
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import json
from tqdm.notebook import tqdm

from scipy.linalg import solve
from scipy.spatial.distance import cdist

from sklearn.base import BaseEstimator, ClassifierMixin, MetaEstimatorMixin, clone
from sklearn.utils.multiclass import unique_labels
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score


from methyldl.deconvolution.evaluation import compute_deconvolution_metrics_np
from methyldl.deconvolution.uxm import (
    load_atlas,
    uxm_deconvolution,
    rearange_uxm_deconvolution_results,
)


%load_ext autoreload
%autoreload 2

In [ ]:
# mapping from cell type names to their corresponding labels in the dataset
with open("../App/labels_dict.json", "r") as f:
    labels_dict = json.load(f)
cell_type_to_label = {v: int(k) for k, v in labels_dict.items()}
N_CELL_TYPES = len(cell_type_to_label)

# constants for accessing the mixtures data
PRED_COLUMNS = [f"prediction_{i}_wavg" for i in range(N_CELL_TYPES)]
SPLIT_TO_IDX = {"train": 0, "val": 1, "test": 2}

## Plotting and training utilities

In [ ]:
@contextmanager
def suppress_many_unique_classes_warning():
    """Suppresses the warning about having many unique classes in the dataset, which occurs when we train SVC with one sample per class."""
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            category=UserWarning,
            message=r"The number of unique classes is greater than 50% of the number of samples*",
        )
        yield

In [ ]:
def softmax(logits: np.ndarray, T: float = 1) -> np.ndarray:
    """Compute the softmax of the logits with temperature scaling."""
    is_logits_1d = logits.ndim == 1
    if is_logits_1d:
        logits = logits.reshape(1, -1)
    scaled_logits = logits / T
    exp_scaled = np.exp(scaled_logits - np.max(scaled_logits, axis=1, keepdims=True))
    result = exp_scaled / np.sum(exp_scaled, axis=1, keepdims=True)
    if is_logits_1d:
        result = result.squeeze()
    return result

In [ ]:
def plot_heatmap(
    matrix: np.ndarray,
    title: str = "Heatmap",
    color_bar_label: str = "Probability",
    vmin: float | None = None,
    vmax: float | None = None,
):
    """Plot a heatmap of the given matrix with cell type names on the axes."""
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(label=color_bar_label)
    plt.xticks(
        ticks=np.arange(matrix.shape[1]),
        labels=[labels_dict[str(i)] for i in range(matrix.shape[1])],
        rotation=90,
    )
    plt.yticks(
        ticks=np.arange(matrix.shape[0]),
        labels=[labels_dict[str(i)] for i in range(matrix.shape[0])],
    )
    plt.title(title)
    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.tight_layout()
    plt.show()

In [ ]:
# dichotomic search to find the highest temperature such that at least `min_pass_count`
# true-class probabilities are above `target_confidence`
def find_temperature_for_confidence(
    logits: np.ndarray,
    true_labels: np.ndarray,
    target_confidence: float = 0.99,
    min_pass_count: int = 1,
    atol: float = 1e-3,
    max_iter: int = 20,
) -> float:
    """
    Find the highest temperature T such that at least `min_pass_count` samples have
    P(true_class) >= target_confidence.
    """
    n_samples = len(true_labels)
    if not (1 <= min_pass_count <= n_samples):
        raise ValueError(
            f"min_pass_count must be in [1, {n_samples}], got {min_pass_count}"
        )

    low, high = 1e-3, 100.0
    best_T = low

    for i in range(max_iter):
        mid_T = (low + high) / 2
        probabilities = softmax(logits, T=mid_T)
        true_class_probs = probabilities[np.arange(n_samples), true_labels]
        pass_count = np.count_nonzero(true_class_probs >= target_confidence)

        if pass_count >= min_pass_count:
            best_T = mid_T
            low = mid_T
        else:
            high = mid_T

        if high - low < atol:
            print(
                f"Dichotomic search converged within tolerance {atol} "
                f"(final range: [{low}, {high}], iter={i})"
            )
            return best_T

    print(
        f"Warning: dichotomic search did not converge within tolerance {atol} "
        f"(final range: [{low}, {high}], iter={max_iter - 1})"
    )
    return best_T

In [ ]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label)

    # plot the cell types names on the x axis, rotated by 90 degrees
    plt.xticks(x_center, cell_type_to_label.keys(), rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title("Predicted Mixtures vs Ground Truth Mixture")
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

## Data Loading

### Pure mixtures

Description of the pkl file :

For every cell_type i:

- df[i][0] - Ground Truth proportions.
- df[i][1][k] - Pandas dataframes with aggregated predictions (k in 0,1,2 means train, validation and test datasets respectfully)
- df[i][2][k] - UXM input matrices and deconvolution results (k in 0,1,2 means train, validation and test datasets respectfully)

In [ ]:
with open("../Data/mixtures/soft_labels_pure_ios.pkl", "rb") as f:
    soft_labels_pure_ios = pickle.load(f)

In [ ]:
# functions to extract data from the loaded pickle file containing the pure mixtures and their corresponding predictions and ground truth labels
def get_pure_prediction_matrix(
    cell_type_name: int | str, split_name: str = "train"
) -> np.ndarray:
    if isinstance(cell_type_name, str):
        cell_type_label = cell_type_to_label[cell_type_name]
    else:
        cell_type_label = cell_type_name
    split_idx = SPLIT_TO_IDX[split_name]
    return soft_labels_pure_ios[cell_type_label][1][split_idx][PRED_COLUMNS].to_numpy()


def get_pure_uxm_results(
    cell_type_name: int | str, split_name: str = "train"
) -> pd.DataFrame:
    if isinstance(cell_type_name, str):
        cell_type_label = cell_type_to_label[cell_type_name]
    else:
        cell_type_label = cell_type_name
    split_idx = SPLIT_TO_IDX[split_name]
    return np.array(soft_labels_pure_ios[cell_type_label][2][split_idx][3])


def get_pure_ground_truth_mixture(cell_type_name: int | str) -> np.ndarray:
    if isinstance(cell_type_name, str):
        cell_type_label = cell_type_to_label[cell_type_name]
    else:
        cell_type_label = cell_type_name
    return np.array(soft_labels_pure_ios[cell_type_label][0])

### Unpure mixtures

The pkl contains a list of 1000 tuples. Each tuple corresponds to a mixture sample and contains the following elements:

- 0: Ground Truth proportions.
- 1: List containing 3 DataFrames (train, validation and test datasets respectfully). Each DataFrame contains the aggregated predictions for each cell type and the corresponding DMR regions.
- 2: List containing 3 tuples (train, validation and test datasets respectfully). Each tuple contains the UXM input matrix for the SVM deconvolution and the corresponding deconvolution results (index 3 for the raw predicted proportions)

In [ ]:
with open("../Data/mixtures/ios_deconvolution_data_hg38_grouped_dmrs_uxm_alligned_1000.pkl", "rb") as f:
    unpure_mixtures_data = pickle.load(f)

In [ ]:
# computing uxm deconvolution results for the unpure mixtures (if not already computed and saved in a pickle file)
file_uxm_unpure_results = "../Data/mixtures/uxm_deconvolution_results_unpure_mixtures.pkl"
if os.path.exists(file_uxm_unpure_results):
    with open(file_uxm_unpure_results, "rb") as f:
        uxm_unpure_results = pickle.load(f)
    print(f"Loaded precomputed UxM deconvolution results for unpure mixtures from {file_uxm_unpure_results}")
else:
    uxm_unpure_results = []
    uxm_atlas, uxm_ref_cells = load_atlas("../Data/UXM_atlas/Atlas.U25.l4.hg38.full.tsv")
    for mixture_idx in tqdm(range(len(unpure_mixtures_data))):
        uxm_unpure_results.append([])
        for split_idx in range(3):
            sf, counts = unpure_mixtures_data[mixture_idx][2][split_idx][:2]
            uxm_raw = uxm_deconvolution(
                atlas=uxm_atlas,
                ref_cells=uxm_ref_cells,
                sf=sf,
                counts=counts,
                sample_names=["pseudo_balk_sample"],
            )[0]
            uxm_aligned = np.array(
                rearange_uxm_deconvolution_results(
                    labels_dict_reversed=cell_type_to_label,
                    uxm_proportions=uxm_raw,
                    ref_cells=uxm_ref_cells,
                )
            )
            uxm_unpure_results[-1].append(uxm_aligned)
    with open(file_uxm_unpure_results, "wb") as f:
        pickle.dump(uxm_unpure_results, f)
    print(f"Saved UxM deconvolution results for unpure mixtures to {file_uxm_unpure_results}")

In [ ]:
# functions to extract the unpure mixtures data from the loaded pickle file, which contains the predictions and ground truth labels for the unpure mixtures
def get_unpure_prediction_matrix(
    mixture_idx: int, split_name: str = "train"
) -> np.ndarray:
    split_idx = SPLIT_TO_IDX[split_name]
    return unpure_mixtures_data[mixture_idx][1][split_idx][PRED_COLUMNS].to_numpy()

def get_unpure_uxm_results(
    mixture_idx: int, split_name: str = "train"
) -> pd.DataFrame:
    split_idx = SPLIT_TO_IDX[split_name]
    return np.array(uxm_unpure_results[mixture_idx][split_idx])

def get_unpure_ground_truth_mixture(mixture_idx: int) -> np.ndarray:
    return np.array(unpure_mixtures_data[mixture_idx][0])

## Data Preprocessing

In [ ]:
# Preparation of the train, validation and test sets for the pure mixtures
def prepare_pure_dataset(split_name: str = "train"):
    X = np.array(
        [
            get_pure_prediction_matrix(cell_type_idx, split_name)
            for cell_type_idx in range(N_CELL_TYPES)
        ]
    )
    y = np.array(
        [
            get_pure_ground_truth_mixture(cell_type_idx)
            for cell_type_idx in range(N_CELL_TYPES)
        ]
    )
    return X, y


# keep the original shape of the data
X_pure_train, y_pure_train = prepare_pure_dataset("train")
X_pure_val, y_pure_val = prepare_pure_dataset("val")
X_pure_test, y_pure_test = prepare_pure_dataset("test")

y_pure_by_split = {
    "train": y_pure_train,
    "val": y_pure_val,
    "test": y_pure_test,
}

# transform the targets to the cell type with the highest proportion in the ground truth mixture
y_pure_train_labels = np.argmax(y_pure_train, axis=1)
y_pure_val_labels = np.argmax(y_pure_val, axis=1)
y_pure_test_labels = np.argmax(y_pure_test, axis=1)

# flatten the matrices
X_pure_train_full = X_pure_train.reshape(X_pure_train.shape[0], -1)
X_pure_val_full = X_pure_val.reshape(X_pure_val.shape[0], -1)
X_pure_test_full = X_pure_test.reshape(X_pure_test.shape[0], -1)

X_pure_full_by_split = {
    "train": X_pure_train_full,
    "val": X_pure_val_full,
    "test": X_pure_test_full,
}

# keep only the diagonal elements of the prediction matrices
X_pure_train_diag = X_pure_train[:, np.arange(N_CELL_TYPES), np.arange(N_CELL_TYPES)]
X_pure_val_diag = X_pure_val[:, np.arange(N_CELL_TYPES), np.arange(N_CELL_TYPES)]
X_pure_test_diag = X_pure_test[:, np.arange(N_CELL_TYPES), np.arange(N_CELL_TYPES)]

In [ ]:
# Preparation of the train, validation and test sets for the unpure mixtures
def prepare_unpure_dataset(split_name: str = "train"):
    X = np.array(
        [
            get_unpure_prediction_matrix(mixture_idx, split_name)
            for mixture_idx in range(len(unpure_mixtures_data))
        ]
    )
    
    y = np.array(
        [
            get_unpure_ground_truth_mixture(mixture_idx)
            for mixture_idx in range(len(unpure_mixtures_data))
        ]
    )
    return X, y

# keep the original shape of the data
X_unpure_train, y_unpure_train = prepare_unpure_dataset("train")
X_unpure_val, y_unpure_val = prepare_unpure_dataset("val")
X_unpure_test, y_unpure_test = prepare_unpure_dataset("test")



y_unpure_by_split = {
    "train": y_unpure_train,
    "val": y_unpure_val,
    "test": y_unpure_test,
}

# transform the targets to the cell type with the highest proportion in the ground truth mixture
y_unpure_train_labels = np.argmax(y_unpure_train, axis=1)
y_unpure_val_labels = np.argmax(y_unpure_val, axis=1)
y_unpure_test_labels = np.argmax(y_unpure_test, axis=1)

# flattens the matrices
X_unpure_train_full = X_unpure_train.reshape(X_unpure_train.shape[0], -1)
X_unpure_val_full = X_unpure_val.reshape(X_unpure_val.shape[0], -1)
X_unpure_test_full = X_unpure_test.reshape(X_unpure_test.shape[0], -1)

X_unpure_full_by_split = {
    "train": X_unpure_train_full,
    "val": X_unpure_val_full,
    "test": X_unpure_test_full,
}

## Modelling

### SVM classification with probabilities as mixture proportions

In [ ]:
def full_svm_experiment(kernel="rbf", data_to_use="full"):
    """Full experiment to train an SVC with probabilities as mixture proportions for the deconvolution task"""
    if data_to_use == "full":
        X_pure_train = X_pure_train_full
        X_pure_val = X_pure_val_full
        X_pure_test = X_pure_test_full
    elif data_to_use == "diag":
        X_pure_train = X_pure_train_diag
        X_pure_val = X_pure_val_diag
        X_pure_test = X_pure_test_diag
    else:
        raise ValueError("data_to_use must be either 'full' or 'diag'")

    with suppress_many_unique_classes_warning():
        # training the SVM classifier
        svm = SVC(kernel=kernel, probability=True)
        svm.fit(X_pure_train, y_pure_train_labels)

        # making predictions on the train, validation and test sets
        train_label_predictions = svm.predict(X_pure_train)
        val_label_predictions = svm.predict(X_pure_val)
        test_label_predictions = svm.predict(X_pure_test)

        train_mixture_predictions = svm.predict_proba(X_pure_train)
        val_mixture_predictions = svm.predict_proba(X_pure_val)
        test_mixture_predictions = svm.predict_proba(X_pure_test)

        train_label_pred_from_proba = train_mixture_predictions.argmax(axis=1)
        val_label_pred_from_proba = val_mixture_predictions.argmax(axis=1)
        test_label_pred_from_proba = test_mixture_predictions.argmax(axis=1)

        train_label_from_inverse_proba = train_mixture_predictions.argmin(axis=1)
        val_label_from_inverse_proba = val_mixture_predictions.argmin(axis=1)
        test_label_from_inverse_proba = test_mixture_predictions.argmin(axis=1)

        # evaluating the predictions
        train_acc = accuracy_score(y_pure_train_labels, train_label_predictions)
        val_acc = accuracy_score(y_pure_val_labels, val_label_predictions)
        test_acc = accuracy_score(y_pure_test_labels, test_label_predictions)

        train_acc_from_proba = accuracy_score(
            y_pure_train_labels, train_label_pred_from_proba
        )
        val_acc_from_proba = accuracy_score(y_pure_val_labels, val_label_pred_from_proba)
        test_acc_from_proba = accuracy_score(y_pure_test_labels, test_label_pred_from_proba)

        train_acc_from_inverse_proba = accuracy_score(
            y_pure_train_labels, train_label_from_inverse_proba
        )
        val_acc_from_inverse_proba = accuracy_score(
            y_pure_val_labels, val_label_from_inverse_proba
        )
        test_acc_from_inverse_proba = accuracy_score(
            y_pure_test_labels, test_label_from_inverse_proba
        )

    train_deconv_metrics = compute_deconvolution_metrics_np(
        train_mixture_predictions, y_pure_train
    )
    val_deconv_metrics = compute_deconvolution_metrics_np(
        val_mixture_predictions, y_pure_val
    )
    test_deconv_metrics = compute_deconvolution_metrics_np(
        test_mixture_predictions, y_pure_test
    )

    return svm, {
        "train": {
            "label_accuracy": train_acc,
            "label_accuracy_from_proba": train_acc_from_proba,
            "inverse_label_accuracy_from_proba": train_acc_from_inverse_proba,
            "deconvolution_metrics": train_deconv_metrics,
        },
        "val": {
            "label_accuracy": val_acc,
            "label_accuracy_from_proba": val_acc_from_proba,
            "inverse_label_accuracy_from_proba": val_acc_from_inverse_proba,
            "deconvolution_metrics": val_deconv_metrics,
        },
        "test": {
            "label_accuracy": test_acc,
            "label_accuracy_from_proba": test_acc_from_proba,
            "inverse_label_accuracy_from_proba": test_acc_from_inverse_proba,
            "deconvolution_metrics": test_deconv_metrics,
        },
    }

In [ ]:
# compute experiments for all combinations of kernel types and data types
# and store the results in a DataFrame
results = []

for kernel_type in ["linear", "rbf"]:
    for data_type in ["full", "diag"]:
        svm_model, metrics = full_svm_experiment(
            kernel=kernel_type, data_to_use=data_type
        )
        results.append(
            {
                "kernel": kernel_type,
                "data_type": data_type,
                "train_label_accuracy": metrics["train"]["label_accuracy"],
                "val_label_accuracy": metrics["val"]["label_accuracy"],
                "test_label_accuracy": metrics["test"]["label_accuracy"],
                "train_label_accuracy_from_proba": metrics["train"][
                    "label_accuracy_from_proba"
                ],
                "val_label_accuracy_from_proba": metrics["val"][
                    "label_accuracy_from_proba"
                ],
                "test_label_accuracy_from_proba": metrics["test"][
                    "label_accuracy_from_proba"
                ],
                "train_inverse_label_accuracy_from_proba": metrics["train"][
                    "inverse_label_accuracy_from_proba"
                ],
                "val_inverse_label_accuracy_from_proba": metrics["val"][
                    "inverse_label_accuracy_from_proba"
                ],
                "test_inverse_label_accuracy_from_proba": metrics["test"][
                    "inverse_label_accuracy_from_proba"
                ],
                "train_deconv_mse": metrics["train"]["deconvolution_metrics"]["mse"],
                "val_deconv_mse": metrics["val"]["deconvolution_metrics"]["mse"],
                "test_deconv_mse": metrics["test"]["deconvolution_metrics"]["mse"],
                "train_deconv_max_error": metrics["train"]["deconvolution_metrics"][
                    "max_error"
                ],
                "val_deconv_max_error": metrics["val"]["deconvolution_metrics"][
                    "max_error"
                ],
                "test_deconv_max_error": metrics["test"]["deconvolution_metrics"][
                    "max_error"
                ],
            }
        )
results_df = pd.DataFrame(results)
results_df.T

In [ ]:
# Examining the gap between scores for the true class and the second best class for each sample in the training set
# Difference per row: diagonal value - second highest value in that row
svm = SVC(kernel="rbf", probability=True)
with suppress_many_unique_classes_warning():
    svm.fit(X_pure_train_full, y_pure_train_labels)
    svm_X_pure_train_pred = svm.decision_function(X_pure_train_full)

n = min(svm_X_pure_train_pred.shape[0], svm_X_pure_train_pred.shape[1])
svm_X_pure_train_pred_n = svm_X_pure_train_pred[:n, :]

diag_values = np.diag(svm_X_pure_train_pred_n)
second_highest_values = np.partition(svm_X_pure_train_pred_n, -2, axis=1)[:, -2]
diag_minus_second = diag_values - second_highest_values
second_best_labels = np.argsort(svm_X_pure_train_pred_n, axis=1)[:, -2]
second_best_labels_names = [labels_dict[str(label)] for label in second_best_labels]
diag_diff_df = pd.DataFrame(
    {
        "diag_value": diag_values,
        "second_highest": second_highest_values,
        "diag_minus_second": diag_minus_second,
        "second_best_label": second_best_labels,
        "second_best_label_name": second_best_labels_names,
    }
).sort_values("diag_minus_second", ascending=True)
diag_diff_df.insert(0, "cell_type_name", [labels_dict[str(i)] for i in range(n)])
diag_diff_df

#### Trying to debug the probability calibration of the SVM classifier

In [ ]:
# plotting the predicted mixtures against the ground truth mixture for a single sample
svm = SVC(kernel="rbf", probability=True)
with suppress_many_unique_classes_warning():
    svm.fit(X_pure_train_full, y_pure_train_labels)
    svm_predicted_mixtures = svm.predict_proba(X_pure_train_full)

In [ ]:
svm.predict_proba(X_pure_train_full)[0, :]

In [ ]:
cell_type_to_label.keys()

In [ ]:
svm.decision_function(X_pure_train_full)[0, :]

In [ ]:
svm.decision_function(X_pure_test_full)[0, :]

In [ ]:
svm.decision_function(X_pure_val_full)[3, :]

In [ ]:
len(svm.n_support_)

In [ ]:
1 / (1 + np.exp(svm.probA_[0] * 38.32370086 + svm.probB_[0]))

In [ ]:
svm.probA_

In [ ]:
def sigmoid_probability(decision_value: float, prob_a: float, prob_b: float) -> float:
    value = 1.0 / (1.0 + np.exp(prob_a * decision_value + prob_b))
    return float(np.clip(value, 1e-7, 1.0 - 1e-7))


def libsvm_multiclass_probability(pairwise_probabilities: np.ndarray) -> np.ndarray:
    """Translate libsvm's multiclass_probability routine to NumPy."""
    n_classes = pairwise_probabilities.shape[0]
    probabilities = np.full(n_classes, 1.0 / n_classes, dtype=float)
    q_matrix = np.zeros((n_classes, n_classes), dtype=float)

    for t in range(n_classes):
        for j in range(t):
            q_matrix[t, t] += pairwise_probabilities[j, t] ** 2
            q_matrix[t, j] = q_matrix[j, t]
        for j in range(t + 1, n_classes):
            q_matrix[t, t] += pairwise_probabilities[j, t] ** 2
            q_matrix[t, j] = (
                -pairwise_probabilities[j, t] * pairwise_probabilities[t, j]
            )
            q_matrix[j, t] = q_matrix[t, j]

    max_iter = max(100, n_classes)
    eps = 0.005 / n_classes

    for _ in range(max_iter):
        q_times_p = q_matrix @ probabilities
        p_q_p = float(probabilities @ q_times_p)
        max_error = np.max(np.abs(q_times_p - p_q_p))
        if max_error < eps:
            break

        for t in range(n_classes):
            diff = (-q_times_p[t] + p_q_p) / q_matrix[t, t]
            probabilities[t] += diff
            normalization = 1.0 + diff
            p_q_p = (p_q_p + diff * (diff * q_matrix[t, t] + 2.0 * q_times_p[t])) / (
                normalization**2
            )
            q_times_p = (q_times_p + diff * q_matrix[:, t]) / normalization
            probabilities /= normalization

    return probabilities


def get_ovo_decision_scores(clf: SVC, X: np.ndarray) -> np.ndarray:
    original_shape = clf.decision_function_shape
    try:
        clf.decision_function_shape = "ovo"
        scores = np.asarray(clf.decision_function(X), dtype=float)
    finally:
        clf.decision_function_shape = original_shape

    n_pairs = len(clf.classes_) * (len(clf.classes_) - 1) // 2
    if scores.ndim == 1 and scores.size == n_pairs:
        return scores[np.newaxis, :]
    if scores.ndim == 1:
        return scores.reshape(-1, 1)
    return scores


def reconstruct_predict_proba_from_scores(
    clf: SVC, X: np.ndarray, flip_score_sign: bool = False
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    ovo_scores = get_ovo_decision_scores(clf, X)
    n_samples = ovo_scores.shape[0]
    n_classes = len(clf.classes_)
    expected_pairs = n_classes * (n_classes - 1) // 2
    assert ovo_scores.shape[1] == expected_pairs
    assert len(clf.probA_) == expected_pairs
    assert len(clf.probB_) == expected_pairs

    pairwise_probability_matrices = np.zeros(
        (n_samples, n_classes, n_classes), dtype=float
    )
    reconstructed_probabilities = np.zeros((n_samples, n_classes), dtype=float)

    for sample_idx in range(n_samples):
        pairwise_probabilities = np.zeros((n_classes, n_classes), dtype=float)
        pair_idx = 0
        for i in range(n_classes):
            for j in range(i + 1, n_classes):
                score = ovo_scores[sample_idx, pair_idx]
                if flip_score_sign:
                    score = -score
                probability_i_vs_j = sigmoid_probability(
                    score, clf.probA_[pair_idx], clf.probB_[pair_idx]
                )
                pairwise_probabilities[i, j] = probability_i_vs_j
                pairwise_probabilities[j, i] = 1.0 - probability_i_vs_j
                pair_idx += 1

        pairwise_probability_matrices[sample_idx] = pairwise_probabilities
        reconstructed_probabilities[sample_idx] = libsvm_multiclass_probability(
            pairwise_probabilities
        )

    return ovo_scores, pairwise_probability_matrices, reconstructed_probabilities

In [ ]:
sample_idx = 0
native_predict_proba = svm.predict_proba(X_pure_train_full)
ovo_scores, pairwise_proba_matrices, reconstructed_direct = (
    reconstruct_predict_proba_from_scores(svm, X_pure_train_full, flip_score_sign=False)
)
_, _, reconstructed_flipped = reconstruct_predict_proba_from_scores(
    svm, X_pure_train_full, flip_score_sign=True
)

direct_l1_error = np.abs(reconstructed_direct - native_predict_proba).sum(axis=1).mean()
flipped_l1_error = (
    np.abs(reconstructed_flipped - native_predict_proba).sum(axis=1).mean()
)
use_flipped_sign = flipped_l1_error < direct_l1_error
reconstructed_predict_proba = (
    reconstructed_flipped if use_flipped_sign else reconstructed_direct
)

comparison_df = pd.DataFrame(
    {
        "class_label": svm.classes_,
        "class_name": [labels_dict[str(label)] for label in svm.classes_],
        "ovo_score_summary": ovo_scores[sample_idx].mean(),
        "native_predict_proba": native_predict_proba[sample_idx],
        "reconstructed_predict_proba": reconstructed_predict_proba[sample_idx],
    }
).sort_values("native_predict_proba", ascending=False)

summary = pd.Series(
    {
        "true_label": y_pure_train_labels[sample_idx],
        "predict": int(svm.predict(X_pure_train_full[[sample_idx]])[0]),
        "argmax_native_predict_proba": int(
            svm.classes_[native_predict_proba[sample_idx].argmax()]
        ),
        "argmax_reconstructed_predict_proba": int(
            svm.classes_[reconstructed_predict_proba[sample_idx].argmax()]
        ),
        "used_flipped_ovo_sign": bool(use_flipped_sign),
        "mean_l1_error_direct": float(direct_l1_error),
        "mean_l1_error_flipped": float(flipped_l1_error),
    }
)

display(summary.to_frame(name="value"))
display(comparison_df.head(10))

In [ ]:
sample_idx = 0
class_idx = 0

pd.DataFrame(
    {
        "opponent_label": svm.classes_,
        "opponent_name": [labels_dict[str(label)] for label in svm.classes_],
        "p_true_class_beats_opponent": pairwise_proba_matrices[sample_idx, class_idx],
    }
).sort_values("p_true_class_beats_opponent")

In [ ]:
cell_type_idx = 0
plot_mixtures_pred_vs_true(
    ground_truth_mixture=get_pure_ground_truth_mixture(cell_type_idx),
    predicted_mixtures=[
        get_pure_uxm_results(cell_type_idx, "train"),
        svm_predicted_mixtures[cell_type_idx],
    ],
    predicted_mixture_labels=["UXM", "SVM (rbf, full matrix)"],
)

Conclusion: the SVM performs well in terms of pure classification, but is terrible in terms of predicting the mixture proportions.

In particular, the predicted mixture proportions are almost uniform, with the proportion of the correct cell_type being the lowest.

This is probably due to the fact that probability calibration requires more data than just one per class.

### LS SVM

#### Implementation of the LS-SVM and training on the pure mixtures

In [ ]:
# Implementation of the Multiclass LS-SVM as described in the paper "Least Squares Support Vector Machine Classifiers" by Suykens and Vandewalle (1999).
class MulticlassLSSVM:
    def __init__(self, gamma=1.0, sigma=1.0, kernel="rbf"):
        """
        Initializes the LS-SVM.
        gamma: Regularization parameter (gamma in the paper).
        sigma: RBF kernel width parameter. Only used when kernel='rbf'.
        kernel: Kernel type to use. Supported values are 'linear' and 'rbf'.
        """
        assert kernel in ["linear", "rbf"], "kernel must be either 'linear' or 'rbf'"
        self.gamma = gamma
        self.sigma = sigma
        self.kernel = kernel
        self.models = []  # Will store (b_i, alpha_i, X_pure_train, y_pure_train_i) for each class

    def _compute_kernel(self, X1, X2):
        """
        Computes the kernel matrix used in the dual LS-SVM system.
        Supported kernels:
        - linear: K(x_k, x_l) = x_k^T x_l
        - rbf: K(x_k, x_l) = exp(-||x_k - x_l||_2^2 / sigma^2)
        """
        if self.kernel == "linear":
            return X1 @ X2.T
        if self.kernel == "rbf":
            dists_sq = cdist(X1, X2, "sqeuclidean")
            return np.exp(-dists_sq / (self.sigma**2))
        raise ValueError(
            f"Unsupported kernel '{self.kernel}'. Expected 'linear' or 'rbf'."
        )

    def fit(self, X, Y):
        """
        Trains the Multiclass LS-SVM.
        X: Training data of shape (N, features)
        Y: One-hot encoded (or multi-output encoded) labels of shape (N, m)
           where m is the number of classes/outputs. Values should be +1 or -1.
        """
        if Y.ndim != 2:
            raise ValueError(
                f"Y must be a 2D array of shape (n_samples, n_classes), got shape {Y.shape}"
            )
        if self.kernel == "rbf" and self.sigma <= 0:
            raise ValueError("sigma must be strictly positive when using the RBF kernel")

        N, m = Y.shape
        self.models = []

        # Calculate the kernel matrix for the training data
        Psi = self._compute_kernel(X, X)

        # Because Y_M and Omega_M are block diagonal, we can solve
        # the linear system independently for each of the m outputs.
        for i in range(m):
            y_i = Y[:, i].reshape(-1, 1)

            # Omega_kl = y_k * y_l * Psi(x_k, x_l) + I / gamma
            # Calculate y_k * y_l matrix
            y_y_T = np.dot(y_i, y_i.T)

            # Element-wise multiplication for Omega + regularization term
            Omega = y_y_T * Psi + np.eye(N) / self.gamma

            # Construct the left-hand side matrix A: [[0, y_T], [y, Omega]]
            A = np.zeros((N + 1, N + 1))
            A[0, 1:] = y_i.T
            A[1:, 0] = y_i.flatten()
            A[1:, 1:] = Omega

            # Construct the right-hand side vector B: [0, 1, ..., 1]^T
            B = np.ones((N + 1, 1))
            B[0, 0] = 0

            # Solve the linear system A * [b; alpha] = B
            solution = solve(A, B)

            # Extract b and alpha
            b_i = solution[0, 0]
            alpha_i = solution[1:, 0].reshape(-1, 1)

            # Store the parameters for this output class
            self.models.append((b_i, alpha_i, X, y_i))

        return self

    def predict(self, X_pure_test):
        """
        Predicts classes for new data points.
        Returns the raw decision values. You can take np.argmax() of the
        output to get the final class prediction if using a 1-vs-all encoding.
        """
        num_test = X_pure_test.shape[0]
        m = len(self.models)
        predictions = np.zeros((num_test, m))

        for i in range(m):
            b_i, alpha_i, X_pure_train, y_i = self.models[i]

            # Calculate kernel between test data and training data
            Psi_test = self._compute_kernel(X_pure_test, X_pure_train)

            # Decision function: sum(alpha_k * y_k * Psi(x, x_k)) + b
            # We compute this in a vectorized manner for all test points
            weighted_Psi = Psi_test * (alpha_i * y_i).T
            predictions[:, i] = np.sum(weighted_Psi, axis=1) + b_i

        return predictions

In [ ]:
lssvm_kernel = "rbf"
lssvm = MulticlassLSSVM(kernel=lssvm_kernel)
lssvm.fit(X_pure_train_full, y_pure_train * 2 - 1)
X_pure_train_pred = lssvm.predict(X_pure_train_full)

# Examining the gap between scores for the true class and the second best class for each sample in the training set
# Difference per row: diagonal value - second highest value in that row
n = min(X_pure_train_pred.shape[0], X_pure_train_pred.shape[1])
X_pure_train_pred_n = X_pure_train_pred[:n, :]

diag_values = np.diag(X_pure_train_pred_n)
second_highest_values = np.partition(X_pure_train_pred_n, -2, axis=1)[:, -2]
diag_minus_second = diag_values - second_highest_values

second_best_labels = np.argsort(X_pure_train_pred_n, axis=1)[:, -2]
second_best_labels_names = [labels_dict[str(label)] for label in second_best_labels]

diag_diff_df = pd.DataFrame(
    {
        "diag_value": diag_values,
        "second_highest": second_highest_values,
        "diag_minus_second": diag_minus_second,
        "second_best_labels": second_best_labels,
        "second_best_labels_names": second_best_labels_names,
    }
)
diag_diff_df.insert(0, "cell_type_name", [labels_dict[str(i)] for i in range(n)])
diag_diff_df.sort_values("diag_minus_second", ascending=True, inplace=True)
diag_diff_df

#### Evaluation of the LS-SVM on the unpure mixtures

In [ ]:
# calibrate the temperature on the pure validation set
lssvm = MulticlassLSSVM(kernel=lssvm_kernel)
with suppress_many_unique_classes_warning():
    lssvm.fit(X_pure_train_full, y_pure_train * 2 - 1)
    X_pure_val_pred = lssvm.predict(X_pure_val_full)
calibrated_T = find_temperature_for_confidence(
    X_pure_val_pred, y_pure_val_labels, target_confidence=0.999, min_pass_count=20
)

unpure_mixture_idx = 2
lssvm_unpure_pred = lssvm.predict(X_unpure_train_full[unpure_mixture_idx].reshape(1, -1)).squeeze()
ground_truth_unpure_mixture = get_unpure_ground_truth_mixture(unpure_mixture_idx)

plot_mixtures_pred_vs_true(
    ground_truth_mixture=ground_truth_unpure_mixture,
    predicted_mixtures=[
        get_unpure_uxm_results(unpure_mixture_idx, "train"),
        softmax(lssvm_unpure_pred.reshape(1, -1), T=calibrated_T).squeeze(),
    ],
    predicted_mixture_labels=[
        "UXM Results",
        f"LS-SVM {lssvm_kernel} Cal. Prob. (T={calibrated_T:.3f})",
    ],
)

In [ ]:
# Evaluate UXM and LS-SVM on all unpure mixtures and store metrics in a single DataFrame

temperatures_to_test = (10**np.linspace(-3, 1, num=10))
additional_temperatures_to_test = 10**np.linspace(-2, -1, num=15)
temperatures_to_test = np.sort(
    np.unique(np.concatenate([temperatures_to_test, additional_temperatures_to_test]))
)
kernels_to_test = ["linear", "rbf"]

split_names = ["train", "val", "test"]
n_mixtures = y_unpure_by_split["train"].shape[0]

# Train one LS-SVM per kernel
lssvm_models = {}
for kernel in kernels_to_test:
    lssvm_model = MulticlassLSSVM(kernel=kernel)
    with suppress_many_unique_classes_warning():
        lssvm_model.fit(X_pure_train_full, y_pure_train * 2 - 1)
    lssvm_models[kernel] = lssvm_model

results = []

# Compute UXM metrics once per split
uxm_preds_by_split = {
    split_name: np.vstack(
        [get_unpure_uxm_results(mixture_idx, split_name) for mixture_idx in range(n_mixtures)]
    )
    for split_name in split_names
}

for split_name in split_names:
    uxm_metrics = compute_deconvolution_metrics_np(
        uxm_preds_by_split[split_name],
        y_unpure_by_split[split_name],
    )
    results.append(
        {
            "model": "UXM",
            "kernel": np.nan,
            "temperature": np.nan,
            "split": split_name,
            **uxm_metrics,
        }
    )

# Compute LS-SVM metrics for each kernel and temperature
for kernel, lssvm_model in lssvm_models.items():
    raw_preds_by_split = {
        split_name: lssvm_model.predict(X_unpure_full_by_split[split_name])
        for split_name in split_names
    }

    for T in temperatures_to_test:
        for split_name in split_names:
            lssvm_probs = softmax(raw_preds_by_split[split_name], T=T)
            lssvm_metrics = compute_deconvolution_metrics_np(
                lssvm_probs,
                y_unpure_by_split[split_name],
            )
            results.append(
                {
                    "model": "LS-SVM",
                    "kernel": kernel,
                    "temperature": T,
                    "split": split_name,
                    **lssvm_metrics,
                }
            )

unpure_metrics_df = pd.DataFrame(results).sort_values(["split", "mse"])

unpure_metrics_df

In [ ]:
# plot deconvolution metrics for the unpure mixtures as a function of the temperature
# for both "rbf" and "linear" kernels; the UXM baseline is plotted as a horizontal line
split = "val"

rbf_metrics = unpure_metrics_df[
    (unpure_metrics_df["model"] == "LS-SVM")
    & (unpure_metrics_df["kernel"] == "rbf")
    & (unpure_metrics_df["split"] == split)
].sort_values("temperature")

linear_metrics = unpure_metrics_df[
    (unpure_metrics_df["model"] == "LS-SVM")
    & (unpure_metrics_df["kernel"] == "linear")
    & (unpure_metrics_df["split"] == split)
].sort_values("temperature")

uxm_metrics = unpure_metrics_df[
    (unpure_metrics_df["model"] == "UXM")
    & (unpure_metrics_df["split"] == split)
]

metrics_to_plot = ["mae", "mse", "kl", "max_error"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

xmin = min(rbf_metrics["temperature"].min(), linear_metrics["temperature"].min())
xmax = max(rbf_metrics["temperature"].max(), linear_metrics["temperature"].max())

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    ax.plot(
        rbf_metrics["temperature"],
        rbf_metrics[metric],
        marker="o",
        label="LS-SVM RBF",
    )
    ax.plot(
        linear_metrics["temperature"],
        linear_metrics[metric],
        marker="s",
        label="LS-SVM Linear",
    )
    ax.hlines(
        uxm_metrics[metric].values[0],
        xmin=xmin,
        xmax=xmax,
        colors="red",
        linestyles="dashed",
        label="UXM",
    )
    ax.set_xscale("log")
    ax.set_xlabel("Temperature (log scale)")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} for Unpure Mixtures - {split} split")
    ax.legend()
    ax.grid()

plt.tight_layout()
plt.show()


In [ ]:
# Select the best LS-SVM model in terms of MSE
best_lssvm_row = unpure_metrics_df[
    (unpure_metrics_df["model"] == "LS-SVM")
    & (unpure_metrics_df["split"] == "val")
].sort_values("mse").iloc[0]

best_lssvm_model = lssvm_models[best_lssvm_row["kernel"]]
best_lssvm_T = best_lssvm_row["temperature"]

# Get all LS-SVM predictions on validation set
lssvm_val_raw_preds = best_lssvm_model.predict(X_unpure_val_full)
lssvm_val_preds = softmax(lssvm_val_raw_preds, T=best_lssvm_T)

# Compute MSE for each mixture
mse_per_mixture = np.mean((lssvm_val_preds - y_unpure_val) ** 2, axis=1)

# Find worst and best mixtures
worst_mixture_idx = np.argmax(mse_per_mixture)
best_mixture_idx = np.argmin(mse_per_mixture)

# Plot worst mixture
plot_mixtures_pred_vs_true(
    ground_truth_mixture=y_unpure_val[worst_mixture_idx],
    predicted_mixtures=[
        get_unpure_uxm_results(worst_mixture_idx, "val"),
        lssvm_val_preds[worst_mixture_idx],
    ],
    predicted_mixture_labels=[
        "UXM Results",
        f"LS-SVM {best_lssvm_row['kernel']} (T={best_lssvm_T:.3f}) - WORST",
    ],
)

# Plot best mixture
plot_mixtures_pred_vs_true(
    ground_truth_mixture=y_unpure_val[best_mixture_idx],
    predicted_mixtures=[
        get_unpure_uxm_results(best_mixture_idx, "val"),
        lssvm_val_preds[best_mixture_idx],
    ],
    predicted_mixture_labels=[
        "UXM Results",
        f"LS-SVM {best_lssvm_row['kernel']} (T={best_lssvm_T:.3f}) - BEST",
    ],
)

print(f"Worst mixture index: {worst_mixture_idx}, MSE: {mse_per_mixture[worst_mixture_idx]:.6f}")
print(f"Best mixture index: {best_mixture_idx}, MSE: {mse_per_mixture[best_mixture_idx]:.6f}")

### ECOC with SVM and LS-SVM as base estimators

In this section, we use the error correcting output codes (ECOC) strategy to train a multi-class LS-SVM.

The idea is to assign one binary code to each class (cell type) and to train one binary LS-SVM for each bit of the code block.

Contrary to classical ECOC where we want to predict a class, here we need to predict a proportion for each class, therefore we will compute an aggregated score for each class by averaging the scores of the corresponding bits of the code block, and then we will apply a softmax to the aggregated scores to obtain the predicted proportions.

#### Example of the idea of the method

Assume 4 classes with the following code matrix.

In [ ]:
example_code_matrix_precursor = [
    "0001",
    "0010",
    "0011",
    "0100",
    "0101",
    "0110",
    "0111",
    "1000",
    "1001",
    "1010",
    "1011",
    "1100",
    "1101",
    "1110",
]
example_code_matrix = np.array([[int(bit) for bit in code] for code in example_code_matrix_precursor]).T
example_code_matrix_m1p1 = example_code_matrix * 2 - 1
example_code_matrix

The assume that we have trained the 14 binary LS-SVMs and that their outputs (in [-1,1]) corresponds perfectly to the mixtures, ie:
- For input mixture A: [1, 0, 0, 0] the output of the 14 binary LS-SVMs is code_matrix[0, :]*2-1
- For input mixture B: [0, 1, 0, 0] the output of the 14 binary LS-SVMs is code_matrix[1, :]*2-1
- For input mixture C: [0, 0.5, 0, 0.5] the output of the 14 binary LS-SVMs is code_matrix[1, :]*2-1*0.5 + code_matrix[3, :]*2-1*0.5
- For input mixture D: [0.25, 0.25, 0.25, 0.25] the output of the 14 binary LS-SVMs is code_matrix[0, :]*2-1*0.25 + code_matrix[1, :]*2-1*0.25 + code_matrix[2, :]*2-1*0.25 + code_matrix[3, :]*2-1*0.25

In [ ]:
for example_mixture in [[1, 0, 0, 0], [0.5, 0.5, 0, 0], [0.25, 0.25, 0.25, 0.25], [0.1, 0.2, 0.3, 0.4], [0.8, 0.1, 0.05, 0.05]]:
    example_mixture_output = np.array(example_mixture) @ example_code_matrix_m1p1
    example_aggregation_results = example_code_matrix_m1p1 @ example_mixture_output
    example_softmax_results = softmax(example_aggregation_results, T=4)
    print(f"Mixture: {example_mixture} -> Aggregation results: {example_aggregation_results} -> Softmax results: {example_softmax_results.round(2)}")

#### Implementation of the ECOC strategy for multi-class LS-SVM

In [ ]:
class CustomECOC(MetaEstimatorMixin, ClassifierMixin, BaseEstimator):
    """
    ECOC meta-estimator for multiclass problems using binary base estimators.

    Each column of the code matrix defines one binary problem. The class-level
    decision scores are obtained by averaging the signed decision values across
    all fitted binary estimators. `predict` returns softmax-normalized scores,
    which can be interpreted as class proportions or probabilities.
    """

    def __init__(self, base_estimator, n_bits, n_classes, matrix_construction_mode="full_random", random_state=42):
        """
        The constructor for the CustomECOC class.

        Args:
            base_estimator: The binary classifier to use for each bit. Must implement fit and decision_function.
            n_bits: The number of bits (columns) in the ECOC code matrix.
            n_classes: The number of classes (rows) in the ECOC code matrix.
            matrix_construction_mode: The method to construct the ECOC code matrix. Currently only supports "full_random".
            random_state: Random seed for reproducibility when constructing the code matrix.
        """
        # perform input validation
        n_bits = int(n_bits)
        if n_bits < int(np.ceil(np.log2(n_classes))):
            raise ValueError(
                f"n_bits={n_bits} is too small for n_classes={n_classes}; must be at least {int(np.ceil(np.log2(n_classes)))}"
            )
        if n_classes < 2:
            raise ValueError("n_classes must be at least 2")
        if not hasattr(base_estimator, "fit"):
            raise TypeError("base_estimator must implement fit")
        if not hasattr(base_estimator, "decision_function"):
            raise TypeError("base_estimator must implement decision_function")

        self.classes_ = np.arange(n_classes)
        self.base_estimator = base_estimator
        self.n_bits = n_bits
        self.n_classes = n_classes
        self.matrix_construction_mode = matrix_construction_mode
        self.random_state = random_state

    def construct_code_matrix(self):
        if self.matrix_construction_mode == "full_random":
            return self._construct_full_random_code_matrix(self.n_classes)
        else:
            raise ValueError(f"Unknown matrix_construction_mode: {self.matrix_construction_mode}")

    def _construct_full_random_code_matrix(self, n_classes):
        """
        Construct the ECOC code matrix.
        """
        rng = np.random.default_rng(self.random_state)

        for _ in range(1000):
            code_matrix = rng.choice([-1.0, 1.0], size=(n_classes, self.n_bits))

            if np.unique(code_matrix, axis=0).shape[0] != n_classes:
                continue

            if np.any(np.abs(code_matrix.sum(axis=0)) == n_classes):
                continue

            return code_matrix

        raise RuntimeError(
            "Failed to construct a valid random ECOC code matrix after 1000 attempts"
        )

    def fit(self, X, y):
        """ 
        Fit the ECOC model according to the given training data.

        Args:
            X: array-like of shape (n_samples, n_features)
                The training input samples.
            y: array-like of shape (n_samples,)
                The target class labels (must be integers from 0 to n_classes-1).
        """
        X, y = check_X_y(X, y)

        assert len(np.unique(y)) <= self.n_classes, f"Number of unique classes in y ({len(np.unique(y))}) exceeds n_classes={self.n_classes}"

        self.code_matrix_ = self.construct_code_matrix()
        self.estimators_ = []

        for bit_idx in range(self.code_matrix_.shape[1]):
            estimator = clone(self.base_estimator)
            y_binary = self.code_matrix_[y, bit_idx]
            estimator.fit(X, y_binary)
            self.estimators_.append(estimator)

        self.n_features_in_ = X.shape[1]
        return self

    def _collect_decision_values(self, X):
        """ 
        Collect the decision values from each binary estimator for the input samples.

        Args:
            X: array-like of shape (n_samples, n_features)
                The input samples for which to collect decision values.
        Returns:
            decision_values: array of shape (n_samples, n_bits)
                The collected decision values from each binary estimator.
        """
        X = check_array(X)
        check_is_fitted(self, ["code_matrix_", "estimators_"])

        decision_values = np.empty((X.shape[0], len(self.estimators_)), dtype=float)

        for bit_idx, estimator in enumerate(self.estimators_):
            bit_scores = np.asarray(estimator.decision_function(X))
            if bit_scores.ndim == 2:
                bit_scores = bit_scores[:, -1]
            decision_values[:, bit_idx] = bit_scores.ravel()

        return decision_values

    def decision_function(self, X):
        """
        Compute the decision function for the input samples.

        Args:
            X: array-like of shape (n_samples, n_features)
                The input samples for which to compute the decision function.
        Returns:
            decision_values: array of shape (n_samples, n_classes)
                The computed decision values for each class.
        """
        decision_values = self._collect_decision_values(X)
        return decision_values @ self.code_matrix_.T / self.code_matrix_.shape[1]

    def predict_proba(self, X):
        raise NotImplementedError()

    def predict(self, X):
        raise NotImplementedError()

    def code_matrix_to_str(self):
        return "\n".join(
            [
                "".join(["1" if bit == 1 else "0" for bit in row])
                for row in self.code_matrix_
            ]
        )

#### ECOC with SVM as base estimator

In [ ]:
# training on pure mixtures
ecoc_svc = CustomECOC(
    base_estimator=SVC(kernel="rbf"),
    n_bits=50,
    n_classes=39,
    matrix_construction_mode="full_random",
    random_state=42
)
ecoc_svc.fit(X_pure_train_full, y_pure_train_labels)

In [ ]:
(ecoc_svc.decision_function(X_pure_train_full).argmax(axis=1) == y_pure_train_labels).mean()

In [ ]:
# visualize the predicted mixtures for the pure training samples using the ECOC SVC
pure_mixture_idx = 2
split_name = "train"
ground_truth_pure_mixture = get_pure_ground_truth_mixture(pure_mixture_idx)
ecoc_svc_raw_prediction = ecoc_svc.decision_function(X_pure_full_by_split[split_name][pure_mixture_idx].reshape(1, -1)).squeeze(0)

plot_mixtures_pred_vs_true(
    ground_truth_mixture=ground_truth_pure_mixture,
    predicted_mixtures=[
        get_pure_uxm_results(pure_mixture_idx, split_name),
        softmax(ecoc_svc_raw_prediction, T=0.05),
    ],
    predicted_mixture_labels=[
        "UXM Results",
        f"ECOC SVC (T=4)",
    ],
)

In [ ]:
# visualize the predicted mixtures for the unpure training samples using the ECOC SVC
unpure_mixture_idx = 2
split_name = "train"
ground_truth_unpure_mixture = get_unpure_ground_truth_mixture(unpure_mixture_idx)
ecoc_svc_raw_prediction = ecoc_svc.decision_function(X_unpure_full_by_split[split_name][unpure_mixture_idx].reshape(1, -1)).squeeze(0)

plot_mixtures_pred_vs_true(
    ground_truth_mixture=ground_truth_unpure_mixture,
    predicted_mixtures=[
        get_unpure_uxm_results(unpure_mixture_idx, split_name),
        softmax(ecoc_svc_raw_prediction, T=0.05),
    ],
    predicted_mixture_labels=[
        "UXM Results",
        f"ECOC SVC (T=4)",
    ],
)

In [ ]:
# compute deconvolution metrics for the ECOC SVC on the specified splits with specified temperatures and compare to UXM results in a DataFrame
# editable parameters
split_names = ["val"]
temperatures_to_test = np.sort(
    np.unique(np.concatenate([10**np.linspace(-2, 0, num=10), []]))
)

n_mixtures = y_unpure_by_split["train"].shape[0]
ecoc_results = []

# UXM baseline
uxm_preds_by_split = {
    split_name: np.vstack(
        [get_unpure_uxm_results(mixture_idx, split_name) for mixture_idx in range(n_mixtures)]
    )
    for split_name in split_names
}

for split_name in split_names:
    uxm_pred = uxm_preds_by_split[split_name]
    uxm_metrics = compute_deconvolution_metrics_np(uxm_pred, y_unpure_by_split[split_name])
    ecoc_results.append(
        {
            "model": "UXM",
            "temperature": np.nan,
            "split": split_name,
            "label_accuracy": float(
                np.mean(
                    np.argmax(uxm_pred, axis=1)
                    == np.argmax(y_unpure_by_split[split_name], axis=1)
                )
            ),
            **uxm_metrics,
        }
    )

# ECOC-SVC results for each temperature
ecoc_raw_by_split = {
    split_name: ecoc_svc.decision_function(X_unpure_full_by_split[split_name])
    for split_name in split_names
}

for T in temperatures_to_test:
    for split_name in split_names:
        ecoc_pred = softmax(ecoc_raw_by_split[split_name], T=T)
        ecoc_metrics = compute_deconvolution_metrics_np(
            ecoc_pred, y_unpure_by_split[split_name]
        )
        ecoc_results.append(
            {
                "model": "ECOC-SVC",
                "temperature": float(T),
                "split": split_name,
                "label_accuracy": float(
                    np.mean(
                        np.argmax(ecoc_pred, axis=1)
                        == np.argmax(y_unpure_by_split[split_name], axis=1)
                    )
                ),
                **ecoc_metrics,
            }
        )

ecoc_metrics_df = pd.DataFrame(ecoc_results).sort_values(["split", "mse", "model"])
ecoc_metrics_df

In [ ]:
# plot in 4 subplots the deconvolution metrics for the ECOC SVC as a function of the temperature a given split, with the UXM baseline as a horizontal line
split = "val"
ecoc_split_metrics = ecoc_metrics_df[
    (ecoc_metrics_df["model"] == "ECOC-SVC") & (ecoc_metrics_df["split"] == split)
].sort_values("temperature")

uxm_split_metrics = ecoc_metrics_df[
    (ecoc_metrics_df["model"] == "UXM") & (ecoc_metrics_df["split"] == split)
]

if ecoc_split_metrics.empty or uxm_split_metrics.empty:
    raise ValueError(f"No ECOC/UXM metrics found for split='{split}'")

metrics_to_plot = ["mae", "mse", "kl", "max_error"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

x_min = ecoc_split_metrics["temperature"].min()
x_max = ecoc_split_metrics["temperature"].max()

for ax, metric in zip(axes, metrics_to_plot):
    ax.plot(
        ecoc_split_metrics["temperature"],
        ecoc_split_metrics[metric],
        marker="o",
        label="ECOC-SVC",
    )
    ax.hlines(
        y=uxm_split_metrics[metric].iloc[0],
        xmin=x_min,
        xmax=x_max,
        colors="red",
        linestyles="dashed",
        label="UXM",
    )
    ax.set_xscale("log")
    ax.set_xlabel("Temperature (log scale)")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} vs Temperature ({split} split)")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

### SVM regression (SVR)

#### Implementation of the SVR and training on the pure mixtures

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted


class SimpleMultiTargetSVR(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        kernel="rbf",
        degree=3,
        gamma="scale",
        coef0=0.0,
        tol=1e-3,
        C=1.0,
        epsilon=0.1,
        shrinking=True,
        cache_size=200,
        verbose=False,
        max_iter=-1,
    ):
        self.kernel = kernel
        self.degree = degree
        self.gamma = gamma
        self.coef0 = coef0
        self.tol = tol
        self.C = C
        self.epsilon = epsilon
        self.shrinking = shrinking
        self.cache_size = cache_size
        self.verbose = verbose
        self.max_iter = max_iter

    def _build_base_estimator(self):
        return SVR(
            kernel=self.kernel,
            degree=self.degree,
            gamma=self.gamma,
            coef0=self.coef0,
            tol=self.tol,
            C=self.C,
            epsilon=self.epsilon,
            shrinking=self.shrinking,
            cache_size=self.cache_size,
            verbose=self.verbose,
            max_iter=self.max_iter,
        )

    def fit(self, X, y):
        X, y = check_X_y(X, y, multi_output=True, y_numeric=True)
        self._y_was_1d_ = y.ndim == 1
        if self._y_was_1d_:
            y = y.reshape(-1, 1)

        self.estimators_ = []
        for i in range(y.shape[1]):
            est = self._build_base_estimator()
            est.fit(X, y[:, i])
            self.estimators_.append(est)

        self.n_features_in_ = X.shape[1]
        self.n_outputs_ = y.shape[1]
        return self

    def predict(self, X, normalize_outputs=False):
        """Normalize outputs will clip the predictions to be non-negative, and then normalize them to sum to 1 for each sample."""
        check_is_fitted(self, "estimators_")
        X = check_array(X)

        preds = np.column_stack([est.predict(X) for est in self.estimators_])

        if normalize_outputs:
            preds = np.clip(preds, a_min=0, a_max=None)
            row_sums = preds.sum(axis=1, keepdims=True)
            preds = np.divide(preds, row_sums, where=row_sums != 0, out=np.zeros_like(preds))

        if self._y_was_1d_:
            return preds.ravel()
        return preds

In [ ]:
svr = SimpleMultiTargetSVR(kernel="linear", C=1, epsilon=1e-4)
svr.fit(X_pure_train_full, y_pure_train)

In [ ]:
def full_svm_experiment(
    epsilon_values: list[float],
    kernel: str = "linear",
    data_to_use: str = "full",
    C: float = 1.0,
    normalize_outputs: bool = True,
):
    """Run a multi-target SVR experiment over several epsilon values and compare against UXM."""
    if len(epsilon_values) == 0:
        raise ValueError("epsilon_values must contain at least one value")

    if data_to_use == "full":
        X_pure_train_used, X_pure_val_used, X_pure_test_used = X_pure_train_full, X_pure_val_full, X_pure_test_full
    elif data_to_use == "diag":
        X_pure_train_used, X_pure_val_used, X_pure_test_used = X_pure_train_diag, X_pure_val_diag, X_pure_test_diag
    else:
        raise ValueError("data_to_use must be either 'full' or 'diag'")

    X_by_split = {
        "train": X_pure_train_used,
        "val": X_pure_val_used,
        "test": X_pure_test_used,
    }
    y_by_split = {
        "train": y_pure_train,
        "val": y_pure_val,
        "test": y_pure_test,
    }

    uxm_predictions_by_split = {
        split_name: np.vstack(
            [
                get_pure_uxm_results(cell_type_idx, split_name)
                for cell_type_idx in range(N_CELL_TYPES)
            ]
        )
        for split_name in X_by_split
    }

    results = []
    models = {}

    for split_name in X_by_split:
        uxm_pred = uxm_predictions_by_split[split_name]
        uxm_metrics = compute_deconvolution_metrics_np(uxm_pred, y_by_split[split_name])
        results.append(
            {
                "model": "UXM",
                "kernel": None,
                "data_type": data_to_use,
                "epsilon": np.nan,
                "split": split_name,
                "label_accuracy": float(
                    np.mean(
                        np.argmax(uxm_pred, axis=1)
                        == np.argmax(y_by_split[split_name], axis=1)
                    )
                ),
                **uxm_metrics,
            }
        )

    for epsilon in epsilon_values:
        svr = SimpleMultiTargetSVR(kernel=kernel, C=C, epsilon=epsilon)
        svr.fit(X_pure_train_used, y_pure_train)
        models[epsilon] = svr

        for split_name, X_split in X_by_split.items():
            y_pred = svr.predict(X_split, normalize_outputs=normalize_outputs)
            deconv_metrics = compute_deconvolution_metrics_np(
                y_pred, y_by_split[split_name]
            )
            results.append(
                {
                    "model": "SVR",
                    "kernel": kernel,
                    "data_type": data_to_use,
                    "epsilon": epsilon,
                    "split": split_name,
                    "label_accuracy": float(
                        np.mean(
                            np.argmax(y_pred, axis=1)
                            == np.argmax(y_by_split[split_name], axis=1)
                        )
                    ),
                    **deconv_metrics,
                }
            )

    results_df = pd.DataFrame(results)
    return models, results_df

In [ ]:
_, results_df = full_svm_experiment(
    epsilon_values=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2,],
    kernel="rbf",
    data_to_use="full",
    C=1.0,
    normalize_outputs=True,
)
results_df

In [ ]:
results_df.loc[results_df["split"] == "test", ].sort_values("mse")

In [ ]:
svr = SimpleMultiTargetSVR(kernel="rbf", C=1, epsilon=1e-3)
svr.fit(X_pure_train_full, y_pure_train)
svr_train_pred = svr.predict(X_pure_train_full, normalize_outputs=True)

# visualize the results with one cell type 
cell_type = "Smooth-Musc" # cell type with low separation between the true class score and the second best class score
# cell_type = "Heart-Cardio" # cell type with high separation between the true class score and the second best class score
cell_type_idx = cell_type_to_label[cell_type]

plot_mixtures_pred_vs_true(
    ground_truth_mixture=get_pure_ground_truth_mixture(cell_type_idx),
    predicted_mixtures=[
        get_pure_uxm_results(cell_type_idx, "train"),
        svr_train_pred[cell_type_idx, :],
    ],
    predicted_mixture_labels=["UXM", "LS-SVM"],
)

#### Evaluation of the SVR on the unpure mixtures

In [ ]:
# Evaluate UXM and SVR on all unpure mixtures (train/val/test) for multiple epsilons and kernels

# User-parameterized epsilon grid (edit as needed)
epsilons_to_test = 10**np.linspace(-4, -2, num=9)
additional_epsilons_to_test = [1e-11]
epsilons_to_test = np.sort(
    np.unique(np.concatenate([epsilons_to_test, additional_epsilons_to_test]))
)
kernels_to_test = ["linear", "rbf"]

# Optional SVR hyperparameters
svr_C = 1.0
normalize_outputs = True

split_names = ["test"]
n_mixtures = y_unpure_by_split["train"].shape[0]

results = []
svr_models = {}

# UXM baseline per split
uxm_preds_by_split = {
    split_name: np.vstack(
        [get_unpure_uxm_results(mixture_idx, split_name) for mixture_idx in range(n_mixtures)]
    )
    for split_name in split_names
}

for split_name in split_names:
    uxm_pred = uxm_preds_by_split[split_name]
    uxm_metrics = compute_deconvolution_metrics_np(uxm_pred, y_unpure_by_split[split_name])
    results.append(
        {
            "model": "UXM",
            "kernel": np.nan,
            "epsilon": np.nan,
            "C": np.nan,
            "split": split_name,
            "label_accuracy": float(
                np.mean(
                    np.argmax(uxm_pred, axis=1)
                    == np.argmax(y_unpure_by_split[split_name], axis=1)
                )
            ),
            **uxm_metrics,
        }
    )

# SVR results per kernel/epsilon
n_loops = len(kernels_to_test) * len(epsilons_to_test) * len(split_names)
with tqdm(total=n_loops, desc="Evaluating SVR models on unpure mixtures") as pbar:
    for kernel in kernels_to_test:
        for epsilon in epsilons_to_test:
            svr = SimpleMultiTargetSVR(kernel=kernel, C=svr_C, epsilon=float(epsilon))
            svr.fit(X_pure_train_full, y_pure_train)
            svr_models[(kernel, float(epsilon))] = svr

            for split_name in split_names:
                y_pred = svr.predict(
                    X_unpure_full_by_split[split_name],
                    normalize_outputs=normalize_outputs,
                )
                svr_metrics = compute_deconvolution_metrics_np(
                    y_pred, y_unpure_by_split[split_name]
                )
                results.append(
                    {
                        "model": "SVR",
                        "kernel": kernel,
                        "epsilon": float(epsilon),
                        "C": svr_C,
                        "split": split_name,
                        "label_accuracy": float(
                            np.mean(
                                np.argmax(y_pred, axis=1)
                                == np.argmax(y_unpure_by_split[split_name], axis=1)
                            )
                        ),
                        **svr_metrics,
                    }
                )
                pbar.update(1)

unpure_svr_metrics_df = pd.DataFrame(results).sort_values(["split", "model", "mse"])
display(unpure_svr_metrics_df)

In [ ]:
split = "val"
rbf_metrics = unpure_svr_metrics_df[
    (unpure_svr_metrics_df["model"] == "SVR")
    & (unpure_svr_metrics_df["kernel"] == "rbf")
    & (unpure_svr_metrics_df["split"] == split)
].sort_values("epsilon")

linear_metrics = unpure_svr_metrics_df[
    (unpure_svr_metrics_df["model"] == "SVR")
    & (unpure_svr_metrics_df["kernel"] == "linear")
    & (unpure_svr_metrics_df["split"] == split)
].sort_values("epsilon")

uxm_metrics = unpure_svr_metrics_df[
    (unpure_svr_metrics_df["model"] == "UXM")
    & (unpure_svr_metrics_df["split"] == split)
]

best_svm_row = unpure_metrics_df[
    (unpure_metrics_df["model"] == "LS-SVM")
    & (unpure_metrics_df["split"] == split)
].sort_values("mse").iloc[0]

metrics_to_plot = ["mae", "mse", "kl", "max_error"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

epsilon_min = min(rbf_metrics["epsilon"].min(), linear_metrics["epsilon"].min())
epsilon_max = max(rbf_metrics["epsilon"].max(), linear_metrics["epsilon"].max())

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    ax.plot(rbf_metrics["epsilon"], rbf_metrics[metric], marker="o", label="SVR RBF")
    ax.plot(
        linear_metrics["epsilon"],
        linear_metrics[metric],
        marker="s",
        label="SVR Linear",
    )
    ax.hlines(
        y=uxm_metrics[metric].iloc[0],
        xmin=epsilon_min,
        xmax=epsilon_max,
        colors="red",
        linestyles="dashed",
        label="UXM",
    )
    ax.hlines(
        y=best_svm_row[metric],
        xmin=epsilon_min,
        xmax=epsilon_max,
        colors="green",
        linestyles="dashdot",
        label=f'Best LS-SVM ({best_svm_row["kernel"]}, T={best_svm_row["temperature"]:.3g})',
    )
    ax.set_xscale("log")
    ax.set_xlabel("Epsilon (log scale)")
    ax.set_ylabel(metric.upper())
    ax.set_title(f"{metric.upper()} vs epsilon ({split} split)")
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# select the SVR with best mse and plot its predictions against the ground truth for one of the unpure mixtures in the validation set
best_svr_row = unpure_svr_metrics_df[
    (unpure_svr_metrics_df["model"] == "SVR")
    & (unpure_svr_metrics_df["split"] == "val")
].sort_values("mse").iloc[0]
best_svr_model = svr_models[(best_svr_row["kernel"], best_svr_row["epsilon"])]

unpure_mixture_idx = 2
svr_unpure_pred = best_svr_model.predict(
    X_unpure_train_full[unpure_mixture_idx].reshape(1, -1),
    normalize_outputs=True,
).squeeze()
ground_truth_unpure_mixture = get_unpure_ground_truth_mixture(unpure_mixture_idx)
plot_mixtures_pred_vs_true(
    ground_truth_mixture=ground_truth_unpure_mixture,
    predicted_mixtures=[
        get_unpure_uxm_results(unpure_mixture_idx, "train"),
        svr_unpure_pred,
    ],
    predicted_mixture_labels=[
        "UXM Results",
        f"SVR {best_svr_row['kernel']} (epsilon={best_svr_row['epsilon']:.3g})",
    ],
)